# Omni NCM: streamed multimodal pipeline (data -> train -> val -> eval)

Level-parallel coarse-to-fine generation across **text + image + audio**, with **video** in the
data path. One schedule, one generator, K forwards per sample regardless of how many
modalities or tokens.

Data plumbing follows `DAPT_v2_stream.ipynb`: a **background producer** streams HF sources,
encodes each sample to K nested code levels with a frozen codec, writes self-describing `.npz`
shards into a rolling buffer, holds out eval samples per group, and signals readiness with a
`EVAL_READY` file. Training reads shards and starts about a minute in.

**Code is cloned from the private repo `adi274903/ncm-omni` into `/content`, not Drive.** Drive
IO stalls the producer and every shard read; only the final checkpoints go back to Drive.

| Cell | What it does |
|---|---|
| 1 | Clone the private repo to `/content`, install `datasets`/`soundfile`/`av` |
| 2 | `run_omni.py check` - 7 model invariants + all four codecs + producer selfcheck, no data |
| 3 | Launch the streaming producer, wait for held-out blocks, keep the handle |
| 4 | Smoke run: 10 steps, s/step, projected hours **and dollars** |
| 5 | Train the level-parallel arm |
| 6 | Train the matched autoregressive arm (identical params, causal mask) |
| 7 | Eval: quality vs NFE table, both arms |
| 8 | Copy checkpoints and the eval table back to Drive |

**The claim under test:** level-parallel reaches the autoregressive arm's quality at NFE = K
instead of NFE = total tokens. If cell 7 shows AR matching quality at NFE <= K, the thesis is
dead and the write-up is the negative result.

### One-time setup for the private clone

Cell 1 needs read access to `adi274903/ncm-omni`. Either one works:

* **Colab secret (recommended)** - paste a GitHub token with `repo` scope into the key icon in
  the left sidebar, name it `GITHUB_TOKEN`, and toggle notebook access on. The token you already
  have stored works: `gh auth token` on your laptop prints it.
* **Authenticated `gh` in the runtime** - if `gh auth status` succeeds in Colab, cell 1 asks it
  for the token itself.

The token is used only inside the clone URL and then stripped from the remote, so it never lands
in `.git/config` or in a committed file.

In [ ]:
import os, sys, subprocess

REPO = 'adi274903/ncm-omni'          # private
PKG  = os.environ.get('NCM_PKG', '/content/ncm_omni')

def _token():
    """Read-only clone access. Colab secret first, then env, then an authenticated gh."""
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t:
            return t.strip()
    except Exception:
        pass
    t = os.environ.get('GITHUB_TOKEN')
    if t:
        return t.strip()
    try:
        return subprocess.check_output(['gh', 'auth', 'token'], text=True).strip()
    except Exception:
        return None

def _askpass(tok):
    """Hand the token to git through GIT_ASKPASS, never through argv or .git/config.

    The old cell pulled with the plain https URL, but the clone deliberately stripped the
    credential out of origin - so a re-run died with `could not read Username`. The token has to
    be re-supplied on every pull, and it must not land in the notebook output or on disk.
    """
    d = os.path.dirname(os.path.abspath(PKG))
    secret = os.path.join(d, '.git-tok')
    with open(secret, 'w') as f:
        f.write(tok)
    os.chmod(secret, 0o600)
    p = os.path.join(d, '.git-askpass.sh')
    with open(p, 'w') as f:
        f.write('#!/bin/sh\ncase "$1" in\n  *sername*) echo x-access-token ;;\n'
                '  *) cat "{}" ;;\nesac\n'.format(secret))
    os.chmod(p, 0o700)
    return p, secret

def _git(args, tok=None):
    env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
    junk = []
    if tok:
        ap, secret = _askpass(tok)
        junk = [ap, secret]
        env.update(GIT_ASKPASS=ap, GIT_ASKPASS_REQUIRE='force')
    try:
        # credential.helper= clears any stale helper so askpass is the only source
        return subprocess.check_output(['git', '-c', 'credential.helper='] + args,
                                       text=True, stderr=subprocess.STDOUT, env=env).strip()
    finally:
        for f in junk:
            if os.path.exists(f):
                os.remove(f)

assert PKG.startswith('/content') or os.environ.get('NCM_PKG'), 'refusing to touch ' + PKG
tok = _token()
url = 'https://github.com/%s.git' % REPO
if os.path.isdir(os.path.join(PKG, '.git')):
    try:
        # Fetch + hard reset, not pull: the checkout is a --depth 1 graft, so a rewound history
        # cannot fast-forward, and run/cfg is rewritten by every invocation. The repo holds no
        # edits worth keeping (run/ is gitignored), so landing exactly on origin/main is right.
        _git(['-C', PKG, 'fetch', '--depth', '1', 'origin', 'main'], tok)
        _git(['-C', PKG, 'reset', '--hard', 'FETCH_HEAD'], tok)
    except subprocess.CalledProcessError as e:
        print('update failed -> re-cloning:', str(e.output).strip().splitlines()[-1])
        subprocess.check_call(['rm', '-rf', PKG])
        _git(['clone', '--depth', '1', url, PKG], tok)
else:
    _git(['clone', '--depth', '1', url, PKG], tok)

assert os.path.exists(os.path.join(PKG, 'core.py')), 'clone landed without core.py'
os.chdir(PKG)
if os.environ.get('NCM_SKIP_PIP') != '1':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                           'datasets', 'soundfile', 'av'])

cfg = os.path.join(PKG, '.git', 'config')
assert 'x-access-token' not in open(cfg).read(), 'credential leaked into .git/config'

import torch
print('repo   ', REPO, '| commit', subprocess.check_output(
    ['git', '-C', PKG, 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('torch  ', torch.__version__, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('workdir', os.getcwd())


In [ ]:
# Invariants first: level k cannot see level k+1, the cross-level knob is wired, every modality
# and level gets a gradient, an absent modality is masked out, AR mode does not leak the future,
# generation costs exactly K forwards, and every codec emits the exact schedule ladder.
!python run_omni.py check


In [ ]:
import time

subprocess.check_call([sys.executable, 'run_omni.py', 'cfg'])          # writes run/cfg/*.json
STREAM = os.path.join(PKG, 'run', 'stream')
LOG    = os.path.join(PKG, 'run', 'producer.log')
os.makedirs(os.path.dirname(LOG), exist_ok=True)

PRODUCER = subprocess.Popen(
    [sys.executable, '-m', 'omni.producer', os.path.join(PKG, 'run', 'cfg', 'produce.json')],
    stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)
print('producer pid', PRODUCER.pid, '- it keeps running after this cell')

t0 = time.time()
while not os.path.exists(os.path.join(STREAM, 'EVAL_READY')):
    if PRODUCER.poll() is not None:
        raise RuntimeError('producer died early:\n' + ''.join(open(LOG).readlines()[-30:]))
    if time.time() - t0 > 1800:
        print(''.join(open(LOG).readlines()[-12:]))
        raise TimeoutError('no held-out blocks in 30 min: check which sources opened')
    time.sleep(5)
print(f'held-out blocks ready after {time.time()-t0:.0f}s')
print(''.join(open(LOG).readlines()[-10:]))


In [ ]:
# 10 steps in, 10 steps out. Prints s/step plus the projected wall clock and cost for the
# configured step budget, so the real run is never a surprise.
!python run_omni.py train --smoke 10


In [ ]:
# Arm 1: level-parallel. Val accuracy is printed per (modality, level) every eval_every steps.
!python run_omni.py train


In [ ]:
# Arm 2: autoregressive, same width/depth/schedule, causal mask over the same raster.
# Same shards, same steps: matched params, matched tokens, one variable changed.
!python run_omni.py train --arm ar


In [ ]:
# Cell: honest level-parallel arm (shifted input, no sibling visibility).
# Same shards, same steps, same params as the AR cell - that is the comparison.
!cd {PKG} && python run_omni.py train --arm strict --steps 300 --budget 4000

## Three arms — and the one that used to cheat

`pack()` used to feed every position **its own target code**. That embedding is in the residual
stream, so the head copied the answer with a linear map: train loss `7.7e-09`, and at generation
(where the input is NULL) accuracy `0.122` — *below* the constant-predictor baseline `0.174`.
That first run measured identity learning, not the architecture.

The input is now shifted so no position can read its own target:

| arm | fed at a position | attention | what it is |
|---|---|---|---|
| `strict` | codes of level k-1, upsampled (level 0: NULL) | levels < k | the honest parallel arm |
| `ar` | code j-1 in raster order | causal | the honest AR arm |
| `block` | its own level's codes | own level | legacy, leaks — do not quote |

Run `strict` and `ar` on the **same shards, same steps, same params**. `eval` prints the
constant-predictor baseline from the shards; anything under it is not a result.


In [ ]:
# The only table that matters: quality vs NFE, greedy, teacher-free, held-out samples.
!python run_omni.py eval


In [ ]:
# Persist: checkpoints and the table are the only things worth keeping off local disk.
import glob, os, shutil

DST = '/content/drive/MyDrive/ncm_omni_runs'
os.makedirs(DST, exist_ok=True)
for src in glob.glob(os.path.join(PKG, 'run', 'train_*', '*.pt')) + \
           glob.glob(os.path.join(PKG, 'run', '*.json')):
    shutil.copy2(src, os.path.join(DST, os.path.basename(src)))
print('copied', len(os.listdir(DST)), 'files to', DST)
print(sorted(os.listdir(DST))[:12])


## Reading cell 7

`NFE` is the number of network evaluations used to produce one sample: the number of levels for
the parallel arm, the number of tokens for the AR arm. A packed sample here is ~506 positions,
so the AR arm's NFE is in the hundreds - that is the whole point of the comparison.

* **Thesis holds** if the parallel arm's `mean acc` at `NFE = K` meets or beats AR's plateau,
  in every modality. Report the ratio.
* **Partial win** is the expected outcome: text and image hold, audio's fine level sags, because
  pooling a spectrogram by 32x destroys the formants. That is a real finding and it is
  publishable - say so instead of hiding it.
* **Thesis dies** if AR reaches the same accuracy at `NFE <= K`, or if the parallel arm's fine
  levels are flat while coarse levels improve (the failure mode `cross_level` exists to test).

Before any of this means anything, the codecs must be real. Today they are: byte codec for text
(finest level decodes exactly), patch-mean pyramid for image and video, log-mel rate pyramid for
audio over a **fixed seeded codebook**. The ladder geometry is the real claim; the codebooks are
what a trained VQ (NCM's text RVQ, Infinity's image VQ, DAC for audio) replaces, and
`omni/codecs.py::build_codec` is the single place to swap them.

Codes are 1-based and 0 is NULL, so `vocab = codebook_size + 1`. Do not "fix" that back to
0-based without also changing `null_code`: real codes would be masked out of the loss silently.